In [2]:
import pandas as pd
import numpy as np

# ==========================
# LOAD DATA
# ==========================

file="../NM Export.xlsx"

xls=pd.ExcelFile(file)

print("Sheets:")
print(xls.sheet_names)

sheet=xls.sheet_names[0]

df=pd.read_excel(
    file,
    sheet_name=sheet,
    dtype=str
)

print("\nOriginal Shape:",df.shape)

# ==========================
# CLEAN COLUMN NAMES
# ==========================

df.columns=(
    df.columns
    .astype(str)
    .str.strip()
    .str.replace("\n"," ",regex=False)
    .str.replace(r"\s+","_",regex=True)
)

# ==========================
# STANDARDIZE EMPTY VALUES
# ==========================

empty_values=[

    "NA",
    "N/A",
    "na",
    "n/a",
    "NULL",
    "null",
    "nan",
    "Not Mentioned",
    "not mentioned",
    "NOT MENTIONED",
    ""
]

df=df.replace(
    empty_values,
    np.nan
)

# ==========================
# REMOVE UNKNOWN COLUMNS
# ==========================

unknown=[]

for col in df.columns:

    c=str(col).lower().strip()

    if (
        "unnamed" in c
        or c==""
        or c=="nan"
    ):
        unknown.append(col)

df=df.drop(
    columns=unknown,
    errors="ignore"
)

print("\nUnknown Columns Removed:")
print(unknown)

# ==========================
# REMOVE 100% EMPTY COLUMNS
# ==========================

drop_cols=[]

for col in df.columns:

    if (
        df[col]
        .fillna("")
        .astype(str)
        .str.strip()
        .eq("")
        .all()
    ):

        drop_cols.append(col)

df=df.drop(
    columns=drop_cols,
    errors="ignore"
)

print("\nRemoved Empty Columns:")
print(drop_cols)

# ==========================
# FIND DESCRIPTION COLUMNS
# ==========================

incident_col=None
event_col=None

for col in df.columns:

    c=(
        col
        .lower()
        .replace("_","")
    )

    if "incidentdetails" in c:

        incident_col=col

    if "incidenteventdetails" in c:

        event_col=col

# ==========================
# REMOVE ROW
# ONLY IF BOTH EMPTY
# ==========================

if incident_col and event_col:

    before=len(df)

    keep=(

        (
            df[incident_col]
            .fillna("")
            .str.strip()
            !=""
        )

        |

        (
            df[event_col]
            .fillna("")
            .str.strip()
            !=""
        )

    )

    df=df[keep]

    print(
        "\nRows Removed:",
        before-len(df)
    )

# ==========================
# REMOVE DUPLICATES
# ==========================

duplicates=df.duplicated().sum()

df=df.drop_duplicates()

print(
    "\nDuplicates Removed:",
    duplicates
)

# ==========================
# ADD SERIAL NUMBER
# ==========================

existing=[]

for col in df.columns:

    c=(
        str(col)
        .lower()
        .replace("_","")
        .replace(":","")
        .replace(" ","")
    )

    if c in [

        "sino",
        "slno",
        "serialno"

    ]:

        existing.append(col)

if existing:

    df=df.drop(
        columns=existing,
        errors="ignore"
    )

df=df.reset_index(
    drop=True
)

df.insert(
    0,
    "SI_No",
    np.arange(
        1,
        len(df)+1
    )
)

print("\nSI_No created")

# ==========================
# MISSING SUMMARY
# ==========================

missing=(

    df
    .replace("",np.nan)
    .isna()
    .sum()

)

missing=missing[
    missing>0
]

print("\nMissing Values Summary:")
print(missing)

# ==========================
# FINAL REPORT
# ==========================

print("\nFinal Shape:")
print(df.shape)

print("\nColumns After Cleaning:")
print(list(df.columns))

# ==========================
# EXPORT
# ==========================

output="cleaned_NM_Export.xlsx"

df.to_excel(
    output,
    index=False
)

print("\nSaved:",output)

Sheets:
['Sheet1']

Original Shape: (9029, 27)

Unknown Columns Removed:
['Unnamed:_26']

Removed Empty Columns:
[]

Rows Removed: 13

Duplicates Removed: 10

SI_No created

Missing Values Summary:
OccurrenceDate                   833
Port                            6226
IncidentDetails                   73
IncidentEventDetails            2166
ImmediateAction                  135
Possibility_Recurrence             1
AreaOfConcern                   1533
DetailedExtentofDamageInjury    2340
DirectCause                     1876
IndirectCause                   8836
RootCause                       8797
Proposedcorrectiveactions        235
Corrective_Action               5020
LeariningPotential              2069
SeverityOfIncident              1170
RevisedIncidentCategory         8416
LessionLearnt                   6329
ReviewComments                  6710
ClosingNote                      712
ClosingDate                     1266
dtype: int64

Final Shape:
(9006, 27)

Columns After Cleaning: